In [1]:
from typing import Any

from dotenv import load_dotenv

load_dotenv("../.env")

True

In [2]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/Users/choyoungrae/Desktop/book-study/books/RAG 마스터 랭체인으로 완성하는 LLM 서비스/rag-master-practice-260326/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import requests
from langchain_core.documents import Document

urls = [
    "https://raw.githubusercontent.com/langchain-kr/langchain-tutorial/refs/heads/main/Ch04.%20Advanced%20Rag/Data/How_to_invest_money.txt"
]

docs = []
for url in urls:
    response = requests.get(url)
    response.raise_for_status()
    response.encoding = "utf-8"
    docs.append(Document(page_content=response.text, metadata={"source": url}))

In [4]:
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

split_docs = recursive_splitter.split_documents(docs)

embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(documents=split_docs, embedding=embeddings)

retriever = vectorstore.as_retriever()

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_text_splitters import CharacterTextSplitter

from langchain_core.runnables import RunnableSerializable

In [6]:
def create_virtual_doc_chain() -> RunnableSerializable:
    system = "당신은 고도로 숙련된 AI입니다."
    user = """
        주어진 질문 '{query}'에 대해 직접적으로 답변하는 가상의 문서를 생성하세요.
        문서의 크기는 {chunk_size} 글자 언저리여야 합니다.
    """.strip()

    prompt = ChatPromptTemplate.from_messages([
        ("system", system),
        ("human", user)
    ])

    llm = ChatOpenAI(model="gpt-4o", temperature=0.2)
    return prompt | llm | StrOutputParser()

In [7]:
from typing import Any

def create_retriever_chain() -> RunnableLambda[Any, list[Document]]:
    return RunnableLambda(lambda x: retriever.invoke(x['virtual_doc']))

def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)

In [8]:
def create_final_response_chain() -> RunnableSerializable:
    final_prompt = ChatPromptTemplate.from_template("""
        다음 정보와 질문을 바탕으로 답변해주세요:

        컨텍스트: {context}

        질문: {question}

        답변:
    """.strip())

    final_llm = ChatOpenAI(model="gpt-4o", temperature=0.2)
    return final_prompt | final_llm

In [9]:
from langchain_core.runnables import RunnableLambda

def print_input_output(input_data, output_data, step_name: str):
    print(f"\n--- {step_name} ---")
    print(f"Input: {input_data}")
    print(f"Output: {output_data}")
    print("-" * 50)

In [11]:
def create_pipeline_with_logging():
    virtual_doc_chain = create_virtual_doc_chain()
    retriever_chain = create_retriever_chain()
    final_response_chain = create_final_response_chain()

    def virtual_doc_step(x):
        result = {
            "virtual_doc": virtual_doc_chain.invoke({
                "query": x["question"],
                "chunk_size": 200
            })
        }
        print_input_output(x, result, "Virtual Doc Generation")
        return {**x, **result}

    def retrieval_step(x):
        result = {
            "retrieved_docs": retriever_chain.invoke(x)
        }
        print_input_output(x, result, "Document Retrieval")
        return {**x, **result}

    def context_formatting_step(x):
        result = {"context": format_docs(x["retrieved_docs"])}
        print_input_output(x, result, "Context Formatting")
        return {**x, **result}

    def final_response_step(x):
        result = final_response_chain.invoke(x)
        print_input_output(x, result, "Final Response Generation")
        return result

    pipeline = (
        RunnableLambda(virtual_doc_step)
        | RunnableLambda(retrieval_step)
        | RunnableLambda(context_formatting_step)
        | RunnableLambda(final_response_step)
    )

    return pipeline

pipeline = create_pipeline_with_logging()

In [12]:
# 예시 질문과 답변
question = "주식 시장의 변동성이 높을 때 투자 전략은 무엇인가요?"
response = pipeline.invoke({"question": question})
print(f"최종 답변: {response.content}")


--- Virtual Doc Generation ---
Input: {'question': '주식 시장의 변동성이 높을 때 투자 전략은 무엇인가요?'}
Output: {'virtual_doc': '주식 시장의 변동성이 높을 때는 신중한 투자 전략이 필요합니다. 첫째, 포트폴리오를 다각화하여 리스크를 분산시키세요. 둘째, 방어주와 같은 안정적인 주식을 고려하세요. 셋째, 현금 비중을 늘려 기회를 대비하세요. 마지막으로, 장기적인 관점을 유지하며 시장의 단기 변동에 흔들리지 않는 것이 중요합니다. 전문가의 조언을 참고하는 것도 도움이 될 수 있습니다.'}
--------------------------------------------------

--- Document Retrieval ---
Input: {'question': '주식 시장의 변동성이 높을 때 투자 전략은 무엇인가요?', 'virtual_doc': '주식 시장의 변동성이 높을 때는 신중한 투자 전략이 필요합니다. 첫째, 포트폴리오를 다각화하여 리스크를 분산시키세요. 둘째, 방어주와 같은 안정적인 주식을 고려하세요. 셋째, 현금 비중을 늘려 기회를 대비하세요. 마지막으로, 장기적인 관점을 유지하며 시장의 단기 변동에 흔들리지 않는 것이 중요합니다. 전문가의 조언을 참고하는 것도 도움이 될 수 있습니다.'}
Output: {'retrieved_docs': [Document(metadata={'source': 'https://raw.githubusercontent.com/langchain-kr/langchain-tutorial/refs/heads/main/Ch04.%20Advanced%20Rag/Data/How_to_invest_money.txt'}, page_content='IX\r\n\r\nMARKET MOVEMENTS OF SECURITIES\r\n\r\n\r\nThere is no question connected with the investment of money more\r\nimporta